<a href="https://colab.research.google.com/github/A-D-Vargas/PI_Mineria_Datos_1/blob/main/A-D-Vargas/PI_Mineria_Datos_1/2_notebooks/01_inspeccion_inicial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Se desarrolló en 01_inspeccion_inicial.ipynb, presentando la estructura general, dimensiones, tipos de datos, valores faltantes, duplicados y observaciones iniciales. Reunimos toda la evidencia para orientar las etapas posteriores.


#**Carga de Datos y Vista Previa**
En esta sección se importan las herramientas necesarias y se cargan los datos para verificar que todo se lea correctamente.

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/A-D-Vargas/datasetproyecto/refs/heads/main/streaming_users_dirty.json"
df_raw = pd.read_json(url)

df_raw.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


---
#**Estructura General y Estadísticas Descriptivas**
Aquí se analiza la estructura interna del archivo (tipos de datos) y se genera un resumen matemático de las columnas numéricas para detectar anomalías físicas inmediatas.

In [ ]:
df_raw.info()
df_raw.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   object 
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   object 
 5   favorite_genre            7920 non-null   object 
 6   last_login_date           7840 non-null   object 
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 510.1+ KB


,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8160.000000,8160.000000,7967.000000,8160.000000
mean,13995.433824,34.096814,1107.346153,1.800980
std,2310.810660,14.511304,5310.442604,11.334969
min,10000.000000,-5.000000,-120.000000,-1.000000
25%,11987.750000,25.000000,489.200000,0.000000
50%,13998.500000,33.000000,757.400000,1.000000
75%,15997.250000,42.000000,1045.700000,1.000000
max,17999.000000,150.000000,99999.000000,150.000000


---
#**Conteo y Porcentaje de Valores Nulos**
Esta celda calcula numéricamente cuánta información falta en todo el set de datos.

In [ ]:
nulos= df_raw.isnull().sum()
porcentaje= (nulos / len(df_raw) * 100).round(2)
print('Cantidades de faltantes: ', nulos)
print('='*35)
print('Porcentaje de nulos:', porcentaje)

Cantidades de faltantes:  user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins     193
country                       0
favorite_genre              240
last_login_date             320
customer_support_tickets      0
dtype: int64
Porcentaje de nulos: user_id                     0.00
age                         0.00
subscription_plan           0.00
monthly_watch_time_mins     2.37
country                     0.00
favorite_genre              2.94
last_login_date             3.92
customer_support_tickets    0.00
dtype: float64


---
#**Detección de Filas Duplicadas**

 Esta celda se encarga de detectar registros idénticos copiados por error en la base de datos.

In [ ]:
duplicados = df_raw.duplicated().sum()
print(f"Número de filas duplicadas: {duplicados}")

# Filtramos las duplicadas y las ordenamos por user_id para ver las parejas/grupos juntos
f_duplicadas = df_raw[df_raw.duplicated(keep=False)].sort_values(by='user_id')

f_duplicadas.head(10)

Número de filas duplicadas: 126


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
37,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8133,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8089,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
52,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
117,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1
8010,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1
8085,10128,19,Básico,638.7,Argentina,Drama,2020-06-17,1
128,10128,19,Básico,638.7,Argentina,Drama,2020-06-17,1
156,10156,43,Básico,592.8,Brasil,Romance,2021-10-25,0
8000,10156,43,Básico,592.8,Brasil,Romance,2021-10-25,0


---
#**Inspección de Valores Únicos y outliers**

## [age]

In [ ]:
df_raw['age'].unique()

array([ 39,  37,  28,  43,  51,  20,  31,  36,  23,  46,  15,  45,  29,
        24,  35,  32,  30,  17,  33,  18,  40,  42,  22,  44,  41,  59,
        26,  25,  27,  53,   0,  55,  34,  47,  16,  21,  13,  38,  56,
        19,  48,  14,  57,  52,  64,  60,  58, 130,  54,  50,  49,  61,
        66,  73, 150,  -5,  68,   4,  69,  62,  72,  63,  67,  65,  76,
        80,  74,  71,  70])

In [ ]:
# 1. Contamos cuántos registros tienen exactamente esos valores atípicos
valores_atipicos = [0, -5, 130, 150]
cantidad_atipicos = df_raw['age'].isin(valores_atipicos).sum()

# 2. Calculamos el porcentaje sobre el total del dataset (8160 registros)
total_registros = len(df_raw)
porcentaje_atipicos = (cantidad_atipicos / total_registros) * 100

# 3. Mostramos los resultados en pantalla
print(f"Cantidad de registros con edades imposibles: {cantidad_atipicos} filas")
print(f"Porcentaje que representan del dataset: {porcentaje_atipicos:.2f}%")

In [ ]:
# Filtramos y mostramos todas las filas que tengan esos 4 valores atípicos en la edad
valores_atipicos = [0, -5, 130, 150]
df_raw[df_raw['age'].isin(valores_atipicos)]

## [customer_support_tickets]

In [ ]:
df_raw['customer_support_tickets'].unique()

array([ 99,   2,   0,   1,   3,   4, 150,  -1,   5])

[monthly_watch_time_mins]
---------------------------------------------------------------------------
- Aquí se analiza qué tan frecuente se repiten ciertos valores de minutos consumidos al mes.

In [ ]:
df_raw['monthly_watch_time_mins'].value_counts()

,count
monthly_watch_time_mins,
0.0,167
-120.0,29
99999.0,20
-1.0,20
50000.0,11
...,...
1375.4,1
346.6,1
477.8,1


## Verificación de inconsistencias categóricas.

#[subscription_plan]
- Se revisa la columna de los planes contratados para ver si los nombres están estandarizados.

In [ ]:
df_raw['subscription_plan'].value_counts()


,count
subscription_plan,
Básico,3450
Estándar,2711
Premium,1519
basico,60
BASICO,52
Basic,52
básico,50
Std,48
Estándar,46


#[country]
- Idéntico al paso anterior, pero aplicado a la ubicación geográfica.

In [ ]:
df_raw['country'].value_counts()

,count
country,
Brasil,1132
Chile,1132
México,1129
Uruguay,1124
Perú,1120
Colombia,1116
Argentina,1087
colombia,27
méxico,25


#[favorite_genre]
- Última celda de control, evalúa las categorías de los gustos de las series o películas.

In [ ]:
df_raw['favorite_genre'].value_counts()

,count
favorite_genre,
Comedia,1112
Drama,1105
Documental,1095
Thriller,1090
Romance,1090
Acción,1082
Crime,1067
Action,22
COMEDIA,19


Al realizar la inspección inicial del dataset streaming_users_dirty.json (8,160 filas en total), se han detectado múltiples anomalías que requieren una etapa de limpieza y transformación. A continuación, se detallan los hallazgos principales clasificados por tipo de problema:

1. **Datos Faltantes (Valores Nulos)**
Se identificaron tres columnas clave con registros ausentes:
* last_login_date: 320 valores nulos.
* favorite_genre: 240 valores nulos.
* monthly_watch_time_mins: 193 valores nulos.

2. **Registros Duplicados**

Se detectaron exactamente 126 filas duplicadas completas (mismo user_id y datos idénticos, como el caso del usuario 10037). Deben ser eliminadas para evitar sesgos en el análisis.

3. **Valores Atípicos y Fuera de Rango (Anomalías Numéricas)**
- **age**: presencia de edades imposibles o biológicamente incoherentes como valores negativos (-5), cero (0), y extremos superiores (130, 150).

* **customer_support_tickets**: valores fuera de contexto como -1, 99 y 150.

* **monthly_watch_time_mins**: valores negativos (-120.0, -1.0) y picos exagerados que parecen errores de sistema o relleno (50000.0, 99999.0).

4. **Inconsistencias de Texto y Categorización (Variables Categóricas)**
Las columnas categóricas presentan problemas de estandarización debido a diferencias de idioma, mayúsculas/minúsculas, acentuación y abreviaturas:

* **subscription_plan**: inconsistencia en el registro de datos por problemas de traducción y tipografía. Se registra como planes distintos lo que en realidad es el mismo (ej. Basic / Básico, Premium / premium).

* **country**: mezcla de nombres completos en español, inglés y códigos de país (ej. Argentina vs. ARG, Colombia vs. COL, Brazil vs. Brasil).

- **favorite_genre**: el mismo género se cuenta como si fueran varios distintos debido a traducciones cruzadas, espacios en blanco y problemas de mayúsculas/minúsculas.